In [1]:
!nvidia-smi

Thu Oct  8 07:14:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from numba import cuda

device = cuda.get_current_device()

print(f"Device Name: {device.name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


In [6]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU (skip very large N)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU Roundtrip (H2D + Kernel + D2H)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU Compute-Only
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

# Run for all sizes
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

print(f"{'N':>12} | {'CPU (ms)':>12} | {'GPU Total (ms)':>15} | {'GPU Kernel (ms)':>16} | Winner")
print("-" * 75)

results = {}
for N in sizes:
    cpu_t, gpu_tot, gpu_ker = benchmark_run(N)
    results[N] = (cpu_t, gpu_tot, gpu_ker)

    if np.isnan(cpu_t):
        winner = "GPU" if gpu_tot < 100 else "—"   # approximate
        cpu_str = "nan"
    else:
        winner = "CPU" if cpu_t < gpu_tot else "GPU"
        cpu_str = f"{cpu_t:.3f}"

    print(f"{N:>12} | {cpu_str:>12} | {gpu_tot:>15.3f} | {gpu_ker:>16.3f} | {winner}")

           N |     CPU (ms) |  GPU Total (ms) |  GPU Kernel (ms) | Winner
---------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


         100 |        0.076 |           1.804 |            0.142 | CPU
        1000 |        0.526 |           0.848 |            0.124 | CPU
       10000 |        4.962 |           0.827 |            0.097 | GPU
      100000 |       24.604 |           0.727 |            0.067 | GPU
     1000000 |      245.139 |           4.766 |            0.124 | GPU
     5000000 |          nan |          13.140 |            0.286 | GPU
    10000000 |          nan |          26.583 |            0.468 | GPU


In [7]:
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)

    d_arr[idx] = 999.0

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4   # 4 × 256 = 1024 threads

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")
print("First 10 elements:", h_result[:10])
print("Last 10 elements:", h_result[-10:])

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0
First 10 elements: [999. 999. 999. 999. 999. 999. 999. 999. 999. 999.]
Last 10 elements: [999. 999. 999. 999. 999. 999. 999. 999. 999. 999.]


In [8]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5 blocks

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<12} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<15}")
print("-" * 55)
for i in range(blocks_per_grid * threads_per_block):
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<12} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<15}")

Global ID    | Block ID   | Thread ID  | Status         
-------------------------------------------------------
0            | 0          | 0          | ACTIVE         
1            | 0          | 1          | ACTIVE         
2            | 0          | 2          | ACTIVE         
3            | 0          | 3          | ACTIVE         
4            | 0          | 4          | ACTIVE         
5            | 0          | 5          | ACTIVE         
6            | 0          | 6          | ACTIVE         
7            | 0          | 7          | ACTIVE         
8            | 1          | 0          | ACTIVE         
9            | 1          | 1          | ACTIVE         
10           | 1          | 2          | ACTIVE         
11           | 1          | 3          | ACTIVE         
12           | 1          | 4          | ACTIVE         
13           | 1          | 5          | ACTIVE         
14           | 1          | 6          | ACTIVE         
15           | 1          | 7   

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:
import numpy as np
from numba import cuda
import time

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]          # DATA RACE!

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])


N = 50_000
h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

print("=" * 60)
print("NAIVE KERNEL (5 trials)")
print("=" * 60)

naive_results = []
for trial in range(1, 6):
    d_total = cuda.to_device(np.array([0.0], dtype=np.float32))

    t0 = time.perf_counter()
    naive_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    t_ms = (time.perf_counter() - t0) * 1000

    result = d_total.copy_to_host()[0]
    deviation = 50000.0 - result
    naive_results.append((result, deviation, t_ms))
    print(f"Trial {trial}: {result:.1f}   (lost: {deviation:.1f})   time: {t_ms:.3f} ms")

print("\n" + "=" * 60)
print("ATOMIC KERNEL")
print("=" * 60)

d_total = cuda.to_device(np.array([0.0], dtype=np.float32))

t0 = time.perf_counter()
atomic_sum_kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
cuda.synchronize()
atomic_time = (time.perf_counter() - t0) * 1000

atomic_result = d_total.copy_to_host()[0]
print(f"Atomic Sum Output: {atomic_result:.1f}")
print(f"Atomic Kernel Time: {atomic_time:.3f} ms")

avg_naive = np.mean([t for _, _, t in naive_results])
print(f"\nAverage Naive time: {avg_naive:.3f} ms")
print(f"Atomic is ~{atomic_time / avg_naive:.1f}x slower than Naive")

NAIVE KERNEL (5 trials)
Trial 1: 2.0   (lost: 49998.0)   time: 368.679 ms
Trial 2: 2.0   (lost: 49998.0)   time: 0.261 ms
Trial 3: 2.0   (lost: 49998.0)   time: 0.266 ms
Trial 4: 2.0   (lost: 49998.0)   time: 0.215 ms
Trial 5: 2.0   (lost: 49998.0)   time: 0.167 ms

ATOMIC KERNEL
Atomic Sum Output: 50000.0
Atomic Kernel Time: 365.978 ms

Average Naive time: 73.918 ms
Atomic is ~5.0x slower than Naive


In [10]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

print(f"Blocks in X: {blocks_2d[0]}")
print(f"Blocks in Y: {blocks_2d[1]}")
print(f"Total Blocks: {blocks_2d[0] * blocks_2d[1]}")

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print(f"Generated 2D matrix shape: {result_canvas.shape}")
print(f"Center value (512, 512): {result_canvas[512, 512]:.6f}")
print(f"Corner value (0, 0):     {result_canvas[0, 0]:.6f}")

Blocks in X: 64
Blocks in Y: 64
Total Blocks: 4096
Generated 2D matrix shape: (1024, 1024)
Center value (512, 512): 1.000000
Corner value (0, 0):     0.000000


In [11]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8') if isinstance(cuda.get_current_device().name, str)
             else cuda.get_current_device().name)
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest


STUDENT_ID = "230103273"

CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: CA76A1E1E8D0259C


'CA76A1E1E8D0259C'